# Machine Learning #01 -  Basic Neural Network in PyTorch 🦥

We will **train a small neural network** to make a
prediction — all in [PyTorch](https://pytorch.org/), one of the most widely used
deep-learning libraries.

We will keep the network **deliberately simple**. The goal is *not* a fancy
architecture, but to understand the **complete PyTorch workflow**:

> **data → tensors → model → loss → training loop → evaluation → interpretation**

## What is PyTorch?

**PyTorch** provides tensors and automatic differentiation for building and training models.

- **Tensors** are multidimensional arrays that can run on CPU or supported accelerators.
- **Autograd** records operations when gradient tracking is enabled and applies the chain rule to compute derivatives. Model parameters require gradients; input features do not need to in this exercise.

Here we write the training loop explicitly so we can connect each operation to the mathematics:

> forward pass → loss → gradients → parameter update

Automatic differentiation is different from estimating derivatives with finite differences.

## Imports and setup

This small model runs comfortably on **CPU**. A Colab GPU is optional; the code uses CUDA when available and otherwise CPU. `torchinfo` is optional and is only used to display a model summary.

In [ ]:
# Core dependencies: torch, numpy, pandas, matplotlib, seaborn, scikit-learn.
# Install missing dependencies in your notebook environment before class.
# torchinfo is optional: the summary cells below include a fallback.

In [ ]:
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

try:
    import torchinfo
except ImportError:
    torchinfo = None

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix

sns.set_theme(style="whitegrid")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(1)  # This tiny model does not benefit from many CPU threads.
print("torch version:", torch.__version__)
print("device:", device)

## The dataset

**Source:** [Species of sloths (Kaggle)](https://www.kaggle.com/datasets/bertiemackie/sloth-species)


| Variable | Meaning |
|---|---|
| `claw_length_cm` | Claw length in cm |
| `size_cm` | Body size in cm |
| `tail_length_cm` | Tail length in cm |
| `weight_kg` | Weight in kg |
| `sub_specie` | One of 6 sub-species  |
| `endangered` | Conservation status |
| `specie` | two-toed vs three-toed |


## Load the data


In [ ]:
CSV_PATH = 'https://github.com/lowoncuties/VSB-FEI-Machine-Learning-Exercises/raw/main/datasets/sloth_data.csv'

df = pd.read_csv(CSV_PATH)
print("Loaded:", CSV_PATH)
df.head()

## A quick look at the data 🔎

We keep exploration **short** here — just enough to understand the task. Two things matter before modelling:

1. **Are the classes balanced?** (Roughly equal numbers of each species?)
2. **Do the features actually separate the classes?**

In [ ]:
print(df["specie"].value_counts())

plt.figure(figsize=(5, 3))
sns.countplot(data=df, x="specie")
plt.title("Number of sloths per species")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.scatterplot(data=df, x="claw_length_cm", y="size_cm",
                hue="specie", alpha=0.6)
plt.title("Claw length vs. body size, coloured by species")
plt.tight_layout()
plt.show()

## Preprocessing

1. Select four numeric feature columns and the target label.
2. Encode the two labels as 0 and 1; inspect the mapping.
3. Create stratified **training / validation / test** splits of **60% / 20% / 20%**.
4. Fit the scaler on the training data only and reuse it for validation and test.

Training data supplies gradients. Validation data selects settings and the checkpoint. Test data is reserved for one final evaluation after the activities. The index column and other metadata are not model inputs.

In [ ]:
feature_cols = ["claw_length_cm", "size_cm", "tail_length_cm", "weight_kg"]
target_col   = "specie"

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4))

for ax, feature in zip(axes, feature_cols):
    sns.histplot(
        data=df,
        x=feature,
        bins=30,
        ax=ax,
        color="steelblue",
    )
    ax.set_title(feature)
    ax.set_xlabel("Value (kg)" if feature == "weight_kg" else "Value (cm)")
    ax.set_ylabel("Count")

fig.suptitle("Distribution of the four input features")
plt.tight_layout()
plt.show()

In [ ]:
X = df[feature_cols].to_numpy(dtype=np.float32)
y_text = df[target_col].to_numpy()
if not np.isfinite(X).all() or pd.isna(y_text).any():
    raise ValueError("Handle missing or non-finite features/labels before modelling.")
if len(np.unique(y_text)) != 2:
    raise ValueError("This notebook requires exactly two target classes.")

In [ ]:
X

In [ ]:
X.shape

In [ ]:
y_text

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)

print("Label mapping:")
for i, name in enumerate(label_encoder.classes_):
    print(f"  {name}  ->  {i}")

In [ ]:
y

In [ ]:
y.shape

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=SEED, stratify=y_temp
)

In [ ]:
X_train.shape, X_test.shape, X_val.shape, y_train.shape, y_val.shape,  y_test.shape

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test  = scaler.transform(X_test)


In [ ]:
X_train[0]

### From NumPy to tensors

Finally we convert the arrays into **PyTorch tensors**.

- Features become **`float32`** tensors (networks work in floating point).
- The target becomes a **`float32`** column of shape `(n, 1)` — this shape matches
  the single output of our network and the loss function we will use.

In [ ]:
X_train.dtype

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
X_val_tensor   = torch.tensor(X_val,   dtype=torch.float32).to(device)
X_test_tensor  = torch.tensor(X_test,  dtype=torch.float32).to(device)

In [ ]:
X_train_tensor.dtype

In [ ]:
X_train.shape

In [ ]:
X_train_tensor.shape

In [ ]:
y_train.dtype

In [ ]:
y_train

In [ ]:
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1).to(device)
y_val_tensor   = torch.tensor(y_val,   dtype=torch.float32).reshape(-1, 1).to(device)
y_test_tensor  = torch.tensor(y_test,  dtype=torch.float32).reshape(-1, 1).to(device)

In [ ]:
y_train_tensor.dtype

In [ ]:
y_train_tensor

## Build the network

Our feed-forward network has one hidden layer:

$$4\text{ inputs}\;\rightarrow\;16\text{ hidden units}\;\rightarrow\;\operatorname{ReLU}\;\rightarrow\;1\text{ logit}.$$

- An affine layer computes $a=Wx+b$ for a column-vector observation. For row batches PyTorch computes $XW^T+b$; its weight shape is `(out_features, in_features)`.
- ReLU, $\max(0,a)$, introduces nonlinearity. The resulting logit is piecewise affine, allowing more complex boundaries than one linear classifier.
- The output is a **raw logit**. During inference, sigmoid converts it to the model's probability of class 1.
- Parameter count: $(4\times16+16)+(16\times1+1)=97$. ReLU adds no parameters.

Do not add a sigmoid layer before `BCEWithLogitsLoss`.

### nn.Sequential

In [ ]:
model = nn.Sequential(
    nn.Linear(4, 16),  # 4 inputs -> 16
    nn.ReLU(),         # non-linearity
    nn.Linear(16, 1),  # 16 -> 1 output (a single logit)
).to(device)

print(model)

In [ ]:
if torchinfo is not None:
    torchinfo.summary(model, input_size=(1, 4), device=device)
else:
    print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

One logit is sufficient for this binary classification setup: sigmoid gives the probability of class 1, and the probability of class 0 is its complement. A two-logit softmax is another possible binary parameterization. For ordinary multiclass classification with cross-entropy, we would normally use one logit per class.

### Loss function and optimizer

For one binary target:

$$\ell=-y\log p-(1-y)\log(1-p),\qquad p=\sigma(z),\qquad \frac{\partial\ell}{\partial z}=p-y.$$

`BCEWithLogitsLoss` accepts **raw logits**, combines sigmoid and BCE stably, and averages over observations by default.

The backward pass computes gradients. The optimizer uses them to update parameters. For plain SGD without momentum or weight decay:

$$\theta\leftarrow\theta-\eta\nabla_\theta L.$$

The baseline uses **Adam**, which keeps moving moment estimates and rescales updates. Its parameter update is not generally equal to the plain SGD formula above.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print("Loss:", criterion)
print("Optimizer:", optimizer.__class__.__name__, "| lr = 0.01")

### The training loop

Each epoch uses the entire training tensor, so there is **one full-batch optimizer step per epoch**:

1. Clear old gradients.
2. Compute logits and mean loss.
3. Compute parameter gradients with `backward()`.
4. Update parameters with `step()`.
5. Measure training and validation loss at the updated parameter state.

Gradients accumulate unless cleared. `backward()` computes gradients; only `step()` updates parameters. We save the checkpoint with the lowest validation loss, not the lowest test loss.

In [ ]:
n_epochs = 100
history = {"train_loss": [], "val_loss": []}
best_val_loss = float("inf")
best_state = None
best_epoch = None

for epoch in range(1, n_epochs + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss = criterion(model(X_train_tensor), y_train_tensor)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        train_loss = criterion(model(X_train_tensor), y_train_tensor).item()
        val_loss = criterion(model(X_val_tensor), y_val_tensor).item()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        best_state = deepcopy(model.state_dict())

model.load_state_dict(best_state)
print(f"Selected epoch: {best_epoch}; validation BCE: {best_val_loss:.4f}")

In [ ]:
plt.figure()
for key, values in history.items():
    plt.plot(values, label=key)
plt.xlabel("Epoch"); plt.ylabel("Loss (BCE)")
plt.legend(); plt.tight_layout(); plt.show()

Inspect the plotted loss history. Loss can fluctuate and is not guaranteed to decrease on every update. Both curves here are measured after the same update. The model has been restored to its best validation checkpoint, which may occur before the final epoch. Predict what a much smaller or larger learning rate might change.

### Validation evaluation

We use **validation data** while comparing models and settings. The test set remains untouched until the end of the activities.

`model.eval()` changes mode-sensitive layers such as dropout and batch normalization; it does not disable autograd. `torch.no_grad()` disables gradient tracking in its context. The current Linear/ReLU network has no mode-dependent layers, but we keep the correct evaluation pattern for later models.

Apply sigmoid for probabilities and threshold at 0.5. Equivalently, a nonnegative logit predicts class 1.

In [ ]:
model.eval()
with torch.no_grad():
    val_logits = model(X_val_tensor)
    val_probs = torch.sigmoid(val_logits)
    val_preds = (val_probs >= 0.5).float()

In [ ]:
y_val_true = y_val_tensor.cpu().numpy().ravel()
y_val_pred = val_preds.cpu().numpy().ravel()
validation_accuracy = accuracy_score(y_val_true, y_val_pred)
print(f"Validation accuracy: {validation_accuracy:.3f}")

### nn.Module

`nn.Sequential` is itself an `nn.Module` and automatically chains its contained modules. A custom `nn.Module` lets us explicitly define the data flow in `forward`, including branches, reused layers, and residual connections. Custom modules can also be placed inside a Sequential container.

In [ ]:
class SlothNet(nn.Module):
    def __init__(self, n_features):
        super().__init__()                     # 1) required first line
        self.fc1 = nn.Linear(n_features, 16)   # 2) define the layers
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x):                       # 3) describe the data flow
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

These definitions describe the same architecture and have the same parameter count. Separately initialized instances generally have different weights and outputs. They become numerically equivalent when their parameter values and execution conditions match.

A custom `forward` is useful for branching, skip connections, or returning intermediate outputs. For today's straightforward stack, either definition is suitable. Architecture definitions control how a prediction is computed; the loss and optimizer remain separate.

Both forms inherit from `nn.Module`, so the same training and evaluation pattern applies. The optimizer must be constructed from the parameters of the particular model we intend to train.

In [ ]:
model_slothnet = SlothNet(4).to(device)
print(model_slothnet)

In [ ]:
if torchinfo is not None:
    torchinfo.summary(model_slothnet, input_size=(1, 4), device=device)
else:
    print("Trainable parameters:", sum(p.numel() for p in model_slothnet.parameters() if p.requires_grad))

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model_slothnet.parameters(), lr=0.01)

In [ ]:
n_epochs = 100
history = {"train_loss": [], "val_loss": []}
best_val_loss = float("inf")
best_state = None
best_epoch = None

for epoch in range(1, n_epochs + 1):
    model_slothnet.train()
    optimizer.zero_grad(set_to_none=True)
    loss = criterion(model_slothnet(X_train_tensor), y_train_tensor)
    loss.backward()
    optimizer.step()

    model_slothnet.eval()
    with torch.no_grad():
        train_loss = criterion(model_slothnet(X_train_tensor), y_train_tensor).item()
        val_loss = criterion(model_slothnet(X_val_tensor), y_val_tensor).item()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        best_state = deepcopy(model_slothnet.state_dict())

model_slothnet.load_state_dict(best_state)
print(f"Selected epoch: {best_epoch}; validation BCE: {best_val_loss:.4f}")

In [ ]:
plt.figure()
for key, values in history.items():
    plt.plot(values, label=key)
plt.xlabel("Epoch"); plt.ylabel("Loss (BCE)")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Evaluate the model that was just trained, on VALIDATION data.
model_slothnet.eval()
with torch.no_grad():
    module_val_logits = model_slothnet(X_val_tensor)
    module_val_probs = torch.sigmoid(module_val_logits)
    module_val_preds = (module_val_probs >= 0.5).float()

In [ ]:
module_val_accuracy = accuracy_score(
    y_val_tensor.cpu().numpy().ravel(),
    module_val_preds.cpu().numpy().ravel(),
)
print(f"SlothNet validation accuracy: {module_val_accuracy:.3f}")

## Setup helper for the activities

Run the following cell to define the reusable training and evaluation helpers. It does not run any experiments. Complete the three activities yourself in the empty cells that follow.

In [ ]:
def build_model(n_features=4, hidden=(16,), activation="relu"):
    layers = []
    previous = n_features
    for width in hidden:
        layers.append(nn.Linear(previous, width))
        if activation == "relu":
            layers.append(nn.ReLU())
        elif activation == "tanh":
            layers.append(nn.Tanh())
        elif activation != "none":
            raise ValueError("activation must be relu, tanh, or none")
        previous = width
    layers.append(nn.Linear(previous, 1))
    return nn.Sequential(*layers)


def train_and_evaluate(X_train, y_train, X_val, y_val, *, hidden=(16,),
                       activation="relu", optimizer_name="adam", lr=.01,
                       epochs=100, seed=42):
    """Fresh full-batch model/optimizer per call; evaluate VALIDATION only.

    Losses are both measured after the update in eval mode. Return the checkpoint
    with the lowest validation BCE, selected among completed training epochs.
    Model selection happens here; held-out test data is deliberately not accepted.
    """
    if epochs < 1:
        raise ValueError("epochs must be positive")
    if X_train.device != X_val.device or y_train.device != X_train.device or y_val.device != X_train.device:
        raise ValueError("All tensors must use the same device")
    if y_train.shape != (len(X_train), 1) or y_val.shape != (len(X_val), 1):
        raise ValueError("BCE targets must have shape (N, 1)")
    torch.manual_seed(seed)
    model = build_model(X_train.shape[1], hidden, activation).to(X_train.device)
    initial_state = deepcopy(model.state_dict())
    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    elif optimizer_name == "sgd":
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    else:
        raise ValueError("optimizer_name must be adam or sgd")
    criterion = nn.BCEWithLogitsLoss()
    history = {"train_loss": [], "val_loss": []}
    best_loss, best_state, best_epoch = float("inf"), None, None
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(X_train), y_train)
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite training loss at epoch {epoch}")
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            train_loss = criterion(model(X_train), y_train).item()
            val_loss = criterion(model(X_val), y_val).item()
        if not np.isfinite([train_loss, val_loss]).all():
            raise FloatingPointError(f"Non-finite post-update loss at epoch {epoch}")
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        if val_loss < best_loss:
            best_loss, best_epoch = val_loss, epoch
            best_state = deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    with torch.no_grad():
        val_accuracy = ((model(X_val) >= 0) == y_val.bool()).float().mean().item()
    return {"model": model, "history": history, "best_epoch": best_epoch,
            "best_val_loss": best_loss, "val_accuracy": val_accuracy,
            "parameter_count": sum(p.numel() for p in model.parameters()),
            "initial_state": initial_state}


def evaluate_once(model, X_test, y_test):
    """Use only after choosing the architecture, settings, epoch, and threshold."""
    model.eval()
    with torch.no_grad():
        logits = model(X_test)
        probs = torch.sigmoid(logits)
        preds = (logits >= 0).long()
        loss = nn.functional.binary_cross_entropy_with_logits(logits, y_test).item()
        accuracy = (preds == y_test.long()).float().mean().item()
    return {"loss": loss, "accuracy": accuracy, "probabilities": probs.cpu().numpy(),
            "predictions": preds.cpu().numpy()}

# Activities

Write your solutions in the empty code cells below. Before each comparison, write a prediction, state which factors are controlled, and then explain the result using validation curves.

The setup helper above creates a fresh model and optimizer for every run. Use it to implement your own experiments.

Do not use the test set to choose a learning rate, architecture, optimizer, or epoch.

## Activity 1 — Learning rate

Compare at least three learning rates with the same architecture, Adam optimizer, initialization, split, and 100-update budget. The deliberately large rate of 1.0 is an experiment, not a recommendation.

The helper records both losses after each update and restores the checkpoint with the lowest **validation** BCE. Its name `train_and_evaluate` refers to validation evaluation; it does not accept test data.

Explain which setting made useful progress within this budget. A large rate need not diverge in every Adam run.

## Activity 2 — Architecture

Compare the baseline with at least two modifications, keeping the split, Adam learning rate 0.01, seed, and update budget fixed. You can change hidden width (for example, 4, 32, or 64), use Tanh, remove the hidden activation, or add a second hidden layer.

Report parameter counts and validation results. Explain why a wider network is not guaranteed to generalize better, and why removing all hidden nonlinearities produces an affine logit function.

## Activity 3 — Optimizer, then final evaluation

Compare SGD at 0.01 and 0.1 with Adam at 0.01. The same nominal learning rate does not imply the same update size for different optimizers. State the learning rate and update budget when interpreting the curves.

Here `torch.optim.SGD` receives the entire training tensor, so it performs full-batch gradient descent without momentum, rather than randomly sampled mini-batch updates.

After completing the declared comparisons, select the configuration/checkpoint with the lowest validation BCE and evaluate it **once on test** at the fixed 0.5 probability threshold. Do not change settings in response to the test result. Record a short explanation of the result and its limitations.